

> # Agentic AI-Based Travel Planning Assistant Using LangChain
**Domain:** Travel & Tourism
**Skills:** Python, LLM Integration, LangChain, API Integration, Streamlit

## 1. Project Context & Objectives
Travelers often switch between multiple websites to manually build itineraries, comparing inconsistent information on flights, hotels, and local attractions. This project solves this by building an intelligent, automated AI Travel Agent using LangChain.
The agent autonomously searches provided datasets and live weather APIs, reasons through the options, and builds an optimized, personalized itinerary based on the user's budget and preferences.



In [16]:
import json
import requests
from datetime import datetime

# Load datasets
with open('flights.json', 'r') as f:
    flights_data = json.load(f)
with open('hotels.json', 'r') as f:
    hotels_data = json.load(f)
with open('places.json', 'r') as f:
    places_data = json.load(f)

def search_flights(source_city: str, destination_city: str) -> list:
    """Filters flights by source and destination, returning the cheapest options."""
    available_flights = [
        f for f in flights_data
        if f['from'].lower() == source_city.lower() and f['to'].lower() == destination_city.lower()
    ]
    return sorted(available_flights, key=lambda x: x['price'])

def search_hotels(city: str, min_stars: int = 3) -> list:
    """Finds hotels in a specific city meeting the minimum star rating."""
    available_hotels = [
        h for h in hotels_data
        if h['city'].lower() == city.lower() and h['stars'] >= min_stars
    ]
    return sorted(available_hotels, key=lambda x: x['price_per_night'])

def search_places(city: str) -> list:
    """Finds top-rated tourist attractions in the destination city."""
    city_places = [
        p for p in places_data
        if p['city'].lower() == city.lower()
    ]
    return sorted(city_places, key=lambda x: x['rating'], reverse=True)

if __name__ == "__main__":
    print("Cheapest Flight to Goa:", search_flights("Bangalore", "Goa")[0])
    print("Top Hotel in Goa:", search_hotels("Goa", min_stars=4)[0])
    print("Top Place in Goa:", search_places("Goa")[0])

Cheapest Flight to Goa: {'flight_id': 'FL0021', 'airline': 'Go First', 'from': 'Bangalore', 'to': 'Goa', 'departure_time': '2025-02-06T07:27:00', 'arrival_time': '2025-02-06T09:27:00', 'price': 5356}
Top Hotel in Goa: {'hotel_id': 'HOT0014', 'name': 'Royal Heritage', 'city': 'Goa', 'stars': 5, 'price_per_night': 1232, 'amenities': ['wifi', 'parking', 'breakfast', 'pool']}
Top Place in Goa: {'place_id': 'PLC0012', 'name': 'Famous Park', 'city': 'Goa', 'type': 'museum', 'rating': 4.5}




> ## 2. Converting Python Logic into LangChain Tools
To allow the AI to autonomously query our data, we convert our basic search functions into LangChain Tools using the `@tool` decorator. We also integrate the live **Open-Meteo API** to pull real-time weather forecasts and a budget calculator.



In [17]:
%%writefile agent_tools.py
import json
import requests
from langchain_core.tools import tool

with open('flights.json', 'r') as f:
    flights_data = json.load(f)
with open('hotels.json', 'r') as f:
    hotels_data = json.load(f)
with open('places.json', 'r') as f:
    places_data = json.load(f)

@tool
def search_flights(source_city: str, destination_city: str) -> dict:
    """Search for the cheapest flight between two cities."""
    available_flights = [
        f for f in flights_data
        if f['from'].lower() == source_city.lower() and f['to'].lower() == destination_city.lower()
    ]
    if not available_flights:
        return {"error": f"No flights found."}
    return sorted(available_flights, key=lambda x: x['price'])[0]

@tool
def search_hotels(city: str, min_stars: int = 3) -> list:
    """Find top hotel recommendations in a city based on star rating."""
    available_hotels = [
        h for h in hotels_data
        if h['city'].lower() == city.lower() and h['stars'] >= min_stars
    ]
    if not available_hotels:
        return [{"error": "No hotels found."}]
    return sorted(available_hotels, key=lambda x: x['price_per_night'])[:2]

@tool
def search_places(city: str) -> list:
    """Discover top-rated tourist attractions in the destination city."""
    city_places = [
        p for p in places_data
        if p['city'].lower() == city.lower()
    ]
    return sorted(city_places, key=lambda x: x['rating'], reverse=True)[:3]

@tool
def get_weather(latitude: float, longitude: float) -> str:
    """Get the 3-day weather forecast for a location using Open-Meteo API."""
    url = f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&daily=temperature_2m_max&timezone=auto"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        temps = data['daily']['temperature_2m_max'][:3]
        return " | ".join([f"Day {i+1}: {temps[i]}°C" for i in range(len(temps))])
    return "Weather data unavailable."

@tool
def estimate_budget(flight_cost: float, hotel_nightly_rate: float, days: int, daily_local_expenses: float = 2500) -> dict:
    """Calculate the estimated total trip budget in INR."""
    total_hotel = hotel_nightly_rate * days
    total_local = daily_local_expenses * days
    return {
        "Flight Cost": f"₹{flight_cost}",
        "Hotel Cost": f"₹{total_hotel}",
        "Food & Travel (Local)": f"₹{total_local}",
        "Estimated Total Budget": f"₹{flight_cost + total_hotel + total_local}"
    }

travel_tools = [search_flights, search_hotels, search_places, get_weather, estimate_budget]

Overwriting agent_tools.py




> ## 3. Building the ReAct / Tool Calling Agent
This script initializes the LLM and gives it a strict system prompt. The prompt forces the agent to use our tools instead of hallucinating facts, and formats the output into the final itinerary required by the project objectives.



In [18]:
%%writefile travel_agent.py
import os
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from agent_tools import travel_tools

# Set your OpenAI API Key
os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert Indian travel planning assistant.
    Your job is to build a complete 3-7 day itinerary based on user preferences.

    You have access to the following tools:
    - search_flights: Find the cheapest flight.
    - search_hotels: Find top-rated accommodations.
    - search_places: Find attractions to visit.
    - get_weather: Get the weather forecast for the destination.
    - estimate_budget: Calculate the total trip cost in INR.

    You must use these tools to gather real data. Do not invent prices or places.

    Format your final response strictly as follows:
    **Your [Number]-Day Trip to [Destination]**

    **Flight Selected:**
    - [Airline] (₹[Price]) – Departs [Source] at [Time]

    **Hotel Booked:**
    - [Hotel Name] (₹[Price]/night, [Stars]-star)

    **Weather:**
    - [Weather Data from tool]

    **Itinerary:**
    - Day 1: [Place 1], [Place 2]
    - Day 2: [Place 3], [Place 4]

    **Estimated Total Budget:**
    - Flight: ₹[Cost]
    - Hotel: ₹[Cost]
    - Food & Travel: ₹[Cost]
    -------------------------------------
    **Total Cost: ₹[Cost]**
    """),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, travel_tools, prompt)
travel_agent_executor = AgentExecutor(agent=agent, tools=travel_tools, verbose=True)

Overwriting travel_agent.py




> ## 4. Streamlit Web Deployment
This code generates the frontend `app.py` file, allowing users to interact with our LangChain agent through a clean web UI.



In [19]:
%%writefile app.py
import streamlit as st
from travel_agent import travel_agent_executor

st.set_page_config(page_title="AI Travel Assistant", layout="centered")
st.title("✈️ AI Travel Planning Assistant")
st.markdown("Enter your travel preferences, and our AI agent will autonomously build your personalized itinerary.")

with st.form("travel_form"):
    query = st.text_area("Describe your ideal trip:",
                         placeholder="e.g., Plan a 3-day trip from Bangalore to Goa. I want a minimum 4-star hotel.")
    submitted = st.form_submit_button("Generate Itinerary")

if submitted and query:
    with st.spinner("Analyzing real-time data and building your itinerary..."):
        try:
            response = travel_agent_executor.invoke({"input": query})
            st.success("Your itinerary is ready!")
            st.markdown(response["output"])
        except Exception as e:
            st.error(f"An error occurred: {str(e)}")

Overwriting app.py
